# Libraries

In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

from sklearn.preprocessing import MinMaxScaler

# Load Data

In [3]:
content_df = pd.read_csv("/content/content.csv")
interactions_df = pd.read_csv("/content/interactions.csv")
users_df = pd.read_excel("/content/users.xlsx")

# Content Embeddings

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

content_df['text'] = (
    content_df['category'].astype(str) + " " +
    content_df['level'].astype(str) + " " +
    content_df['description'].astype(str)
)

content_embeddings = model.encode(content_df['text'].tolist())

content_sim = cosine_similarity(content_embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Collaborative Filtering

In [5]:
user_item = interactions_df.pivot_table(
    index='user_id',
    columns='content_id',
    values='rating',
    fill_value=0
)

svd = TruncatedSVD(n_components=50, random_state=42)

user_factors = svd.fit_transform(user_item)
item_factors = svd.components_

collab_pred = np.dot(user_factors, item_factors)

collab_df = pd.DataFrame(
    collab_pred,
    index=user_item.index,
    columns=user_item.columns
)

# Hybrid Recommendation System

In [8]:
def hybrid_recommend(user_id, top_n=5):

    all_items = content_df['content_id'].values

    # ======================
    # Content Score
    # ======================
    content_scores = pd.Series(0, index=all_items, dtype=float)

    user_history = interactions_df[
        interactions_df['user_id'] == user_id
    ]['content_id'].values

    indices = content_df[content_df['content_id'].isin(user_history)].index

    if len(indices) > 0:
        scores = np.sum(content_sim[indices], axis=0)
        content_scores = pd.Series(scores, index=all_items)

    # ======================
    # Collaborative Score (SVD)
    # ======================
    if user_id in collab_df.index:
        collab_scores_raw = collab_df.loc[user_id]
        # Reindex collab_scores to align with all_items and fill missing with 0
        collab_scores = collab_scores_raw.reindex(all_items, fill_value=0)
    else:
        collab_scores = pd.Series(0, index=all_items)

    # ======================
    # Normalize
    # ======================
    scaler = MinMaxScaler()

    content_norm = scaler.fit_transform(content_scores.values.reshape(-1,1)).flatten()
    collab_norm = scaler.fit_transform(collab_scores.values.reshape(-1,1)).flatten()

    # ======================
    # Final Score
    # ======================
    final_score = 0.5 * content_norm + 0.5 * collab_norm

    result = pd.DataFrame({
        "content_id": all_items,
        "score": final_score
    })

    result = result.merge(content_df, on="content_id")
    result = result.sort_values("score", ascending=False)

    return result.head(top_n)

# Test Recommendation

In [9]:
recommendations = hybrid_recommend(user_id=1)
print(recommendations[['content_id','title','score']])

      content_id                                 title     score
1970        1971                  Python for Beginners  0.852505
137          138        Introduction to Cyber Security  0.786913
1191        1192            Power BI for Data Analysis  0.639548
1628        1629  Artificial Intelligence Fundamentals  0.630461
914          915              Cloud Computing with AWS  0.627977


# Evaluation

In [10]:
def evaluate(user_id, k=5):

    recs = hybrid_recommend(user_id)['content_id'].tolist()

    actual = interactions_df[
        (interactions_df['user_id'] == user_id) &
        (interactions_df['rating'] >= 3)
    ]['content_id'].tolist()

    if len(actual) == 0:
        return None

    precision = len(set(recs[:k]) & set(actual)) / k
    recall = len(set(recs[:k]) & set(actual)) / len(actual)

    f1 = 0 if precision+recall == 0 else 2*(precision*recall)/(precision+recall)

    return precision, recall, f1

# Run Evaluation for Many Users

In [12]:
users = interactions_df['user_id'].unique()[:50]

results = []

for u in users:
    res = evaluate(u)

    if res is not None:
        results.append(res)

results = np.array(results)

print("📊 FINAL RESULTS")
print("Precision@5:", results[:,0].mean())


📊 FINAL RESULTS
Precision@5: 0.7199999999999999


# Run Evaluation for One User

In [14]:
user_id = 1
k = 5

recs = hybrid_recommend(user_id)['content_id'].tolist()

actual = interactions_df[
    (interactions_df['user_id'] == user_id) &
    (interactions_df['rating'] >= 3)
]['content_id'].tolist()

precision = len(set(recs[:k]) & set(actual)) / k if k else 0

recall = len(set(recs[:k]) & set(actual)) / len(actual) if len(actual) > 0 else 0

f1 = 0 if (precision + recall) == 0 else 2 * (precision * recall) / (precision + recall)

print(f"📊 Evaluation for User {user_id}")
print("Precision@5:", precision)


📊 Evaluation for User 1
Precision@5: 0.8
